# 02 — Fit Surrogates

Train surrogate models on sweep data from each partial model.

**Prerequisites**: Run `bayesmm run` on all 4 model specs first (see notebook 01).

> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: turn a parameter sweep into a fast probabilistic surrogate per model.
- **Secondary scientific**: explain why the metamodel needs surrogates at all.

## Why surrogates

Joint inference has to evaluate each model thousands of times. The KS model alone takes
seconds per evaluation, so sampling it directly inside MCMC is hopeless.

A surrogate is a cheap probabilistic stand-in fit to a sweep: it predicts the model's
output at unseen inputs *and reports its own uncertainty*. That second part is what
makes it usable in a Bayesian metamodel — the joint posterior needs to know how much to
trust each surrogate, and a point-estimate emulator cannot say.

**Requires a backend**: `pymc` for `pymc_gp`, `sbi` for `sbi_npe`.

In [ ]:
import json
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Run parameter sweeps

In [ ]:
for spec in sorted(SPECS.glob("model.*.json")):
    print(f"Running sweep: {spec.name}")
    r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "run", str(spec)], cwd=str(ROOT),
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Done")
    else:
        print(f"  FAILED: {r.stderr.strip()[:200]}")

## Fit surrogates

In [ ]:
for spec in sorted(SPECS.glob("surrogate.*.json")):
    print(f"Fitting surrogate: {spec.name}")
    r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "fit", str(spec)], cwd=str(ROOT),
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Done")
    else:
        print(f"  FAILED: {r.stderr.strip()[:200]}")

## List trained surrogates

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Final check

In [ ]:
assert ROOT.is_dir()
print("[NB02 self-check OK]")